In [ ]:
import math
import random
import matplotlib.pyplot as plt
import time
import numpy as np
from decimal import *
getcontext().prec=36
class g_rho():
    def __init__(self,number,s,r):
        self.number = number
        self.r = r
        self.s = s
        self.g_rho = [ [ ((k+2)*(k+1)/2+(k+1)*k/2)**(-0.501-2*self.r*self.s)/5 for j in range(2*k+1)] for k in range(self.number)]
        self.C_K_2_List = []
        self.C_K_1_List = [1]
        
    def Centeral_C_K_update(self,x,k):
        if k == 0:
            self.C_K_List = [1]
        else:
            self.C_K_List = []
            self.C_K_2_List.append(0)
            j = k 
            lambda_j = k-j+1/2
            for k_1,k_2 in zip(self.C_K_1_List,self.C_K_2_List):
                self.C_K_List.append((2*(j+lambda_j-1)*x*k_1)/j - (j+2*lambda_j-2)*k_2/j)
                j = j - 1
                lambda_j = k-j+1/2
            self.C_K_List.append(1)
            self.C_K_2_List = self.C_K_1_List[:]
            self.C_K_1_List = self.C_K_List[:]
            
    def grho_cal(self,an_theta, an_phi):
        self.Computation_Yn_grho(an_theta,an_phi)
        return sum([sum([ a*b for a,b in zip(A,B)]) for A,B in zip(self.g_rho,self.Y_n_g_rho)])

    def Computation_Yn_grho(self,an_theta,an_phi):
        factorial_k = 1
        self.Y_n_g_rho = [[1]]
        self.C_K_2_List = []
        self.C_K_1_List = [1]
        for k in range(1,len(self.g_rho)):
            factorial_k *= k
            factorial_k_j_plus = [factorial_k]
            factorial_k_j_subtrac = [factorial_k]
            db_factorial = [1]
            sin_theta_k = [1]
            for j in range(1,k+1):
                sin_theta_k.append(sin_theta_k[-1]*math.sin(an_theta))
                factorial_k_j_plus.append(factorial_k_j_plus[-1]*(k+j))
                factorial_k_j_subtrac.append(factorial_k_j_subtrac[-1]/(k-j+1))
                db_factorial.append((2*j-1)*db_factorial[-1])
            self.Centeral_C_K_update(math.cos(an_theta),k)
            Y_List = [math.sqrt(2*(2*k+1)*factorial_k_j_subtrac[j]/factorial_k_j_plus[j])*db_factorial[j]*sin_theta_k[j]*self.C_K_List[j]  for j in range(k+1)]
            Y_List[0] /= math.sqrt(2)
            self.Y_n_g_rho.append([Y_List[j]*math.cos(j*an_phi) for j in range(k+1)]+[Y_List[j]*math.sin(j*an_phi) for j in range(1,k+1)])
    
    def construct_data(self):
        X = np.random.normal(size=(3))
        x,y,z = X/math.sqrt(sum(X**2))
        an_theta, an_phi = self.cartesian_to_spherical(x, y, z)
        Y = self.grho_cal(an_theta, an_phi)
        Z = Y + random.gauss(0,0.2)
        return [[an_theta, an_phi],Z,Y]

    def cartesian_to_spherical(self,x, y, z):
        theta = np.arccos(np.clip(z, -1.0, 1.0))
        phi = np.arctan2(y, x)  # returns in (-pi, pi]
        if  phi < 0:
            phi += 2*np.pi
        return theta, phi

DATA = []
s = 1.
r = 1.
G_rho = g_rho(11,s,r)
N = [int(10**(i/5)) for i in range(1,35)]
for i in range(6309630):
    DATA.append(G_rho.construct_data())
    if i in N:
        print(i)
      

In [ ]:
import numpy as np
import math
import random
import time
class Kernel_SGD():
    def __init__(self,class_num,d,a,loss,kernel,sigma,number,r,s):
        self.class_num = class_num
        self.d = d
        self.Alpha_last = np.zeros((1,self.class_num))
        self.Alpha_ave = np.zeros((1,self.class_num))
        self.X = [np.zeros(self.d)]
        self.s1 = s
        self.cc_B4 = -math.pi**4/48
        self.cc_B2 = math.pi**2/4
        self.loss = loss
        self.kernel = kernel
        self.sigma = sigma
        self.number = number
        self.r = r
        self.s = s
        self.g_rho = [ [ ((k+2)*(k+1)/2+(k+1)*k/2)**(-0.501-2*self.r*self.s)/5 for j in range(2*k+1)] for k in range(number)]
        self.C_K_2_List = []
        self.C_K_1_List = [1]
        


    def kernel_com(self,x1,x2):
        t = x1-x2
        if self.kernel == 'Matern52':
            tau = np.sqrt(np.dot(t,t))
            value = (1+np.sqrt(5)*(tau)+5*(tau**2)/3)*np.exp(-np.sqrt(5)*tau)
        elif self.kernel == 'Gauss':
            tau = np.dot(t,t)
            value = np.exp(-tau/2)
        elif self.kernel == 'Matern32':
            tau = np.sqrt(np.dot(t,t))
            value = (1+np.sqrt(3)*tau)*np.exp(-np.sqrt(3)*tau)
        elif self.kernel == 'B4':
            tau = np.sqrt(np.dot(t,t))
            fraction_xy = tau-int(tau)
            value = (((fraction_xy-2)*fraction_xy+1)*fraction_xy**2-1/30)*self.cc_B4
        return value

    def computation_f_last(self,x):
        S = np.zeros(self.class_num)
        if len(self.Alpha_last) > 0:
            A = np.array([ self.kernel_com(a,x) for a in self.X])
            S = np.dot(A,self.Alpha_last)
        return S

    def computation_f_ave(self,x):
        S = np.zeros(self.class_num)
        if len(self.Alpha_ave) > 0:
            A = np.array([ self.kernel_com(a,x) for a in self.X])
            S = np.dot(A,self.Alpha_ave)
        return S

    def derivative_loss(self,X,Y):
        if self.loss == 'least square':
            derivative = (self.computation_f_last(X)-Y)
        elif self.loss == 'logistic':
            term = np.exp(Y*self.computation_f_last(X))
            derivative = - Y / (1+term)
        elif self.loss == 'Cauchy':
            term = self.computation_f_last(X)-Y
            derivative = term/(self.sigma**2+term**2/2)
        elif self.loss == 'Huber':
            term = self.computation_f_last(X)-Y
            derivative = term/np.sqrt(term**2+1)
        elif self.loss == 'Welsch':
            term = self.computation_f_last(X)-Y
            derivative = np.exp(-term**2/(2*self.sigma**2))*(term/self.sigma**2)
        return derivative
            
    def updata(self,x,y,epoch):
        self.gamma = self.gamma0*((epoch)**(-self.t))
        coe = -self.gamma*self.derivative_loss(x,y)
        self.Alpha_last = np.append(self.Alpha_last,coe.reshape(1,self.class_num),axis =0)
        self.Alpha_ave = np.append(self.Alpha_ave, np.zeros((1,self.class_num)), axis = 0)
        self.Alpha_ave = (epoch*self.Alpha_ave +self.Alpha_last)/(epoch+1)
        self.X.append(x)


    def Centeral_C_K_update(self,x,k):
        if k == 0:
            self.C_K_List = [1]
        else:
            self.C_K_List = []
            self.C_K_2_List.append(0)
            j = k 
            lambda_j = k-j+1/2
            for k_1,k_2 in zip(self.C_K_1_List,self.C_K_2_List):
                self.C_K_List.append((2*(j+lambda_j-1)*x*k_1)/j - (j+2*lambda_j-2)*k_2/j)
                j = j - 1
                lambda_j = k-j+1/2
            self.C_K_List.append(1)
            self.C_K_2_List = self.C_K_1_List[:]
            self.C_K_1_List = self.C_K_List[:]

    def grho_cal(self,an_theta, an_phi):
        self.Computation_Yn_grho(an_theta,an_phi)
        return sum([sum([ a*b for a,b in zip(A,B)]) for A,B in zip(self.g_rho,self.Y_n_g_rho)])

    def Computation_Yn_grho(self,an_theta,an_phi):
        factorial_k = 1
        self.Y_n_g_rho = [[1]]
        self.C_K_2_List = []
        self.C_K_1_List = [1]
        for k in range(1,len(self.g_rho)):
            factorial_k *= k
            factorial_k_j_plus = [factorial_k]
            factorial_k_j_subtrac = [factorial_k]
            db_factorial = [1]
            sin_theta_k = [1]
            for j in range(1,k+1):
                sin_theta_k.append(sin_theta_k[-1]*math.sin(an_theta))
                factorial_k_j_plus.append(factorial_k_j_plus[-1]*(k+j))
                factorial_k_j_subtrac.append(factorial_k_j_subtrac[-1]/(k-j+1))
                db_factorial.append((2*j-1)*db_factorial[-1])
            self.Centeral_C_K_update(math.cos(an_theta),k)
            Y_List = [math.sqrt(2*(2*k+1)*factorial_k_j_subtrac[j]/factorial_k_j_plus[j])*db_factorial[j]*sin_theta_k[j]*self.C_K_List[j]  for j in range(k+1)]
            Y_List[0] /= math.sqrt(2)
            self.Y_n_g_rho.append([Y_List[j]*math.cos(j*an_phi) for j in range(k+1)]+[Y_List[j]*math.sin(j*an_phi) for j in range(1,k+1)])
 
    def construct_data(self):
        X = np.random.normal(size=(3))
        x,y,z = X/math.sqrt(sum(X**2))
        an_theta, an_phi = self.cartesian_to_spherical(x, y, z)
        Y = self.grho_cal(an_theta, an_phi)
        Z = Y + random.gauss(0, 0.2)
        return [np.array([x,y,z]),Z,Y]
    '''
    def construct_data(self):
        return DATA[self.epoch]
    '''
    def cartesian_to_spherical(self,x, y, z):
        theta = np.arccos(np.clip(z, -1.0, 1.0))
        phi = np.arctan2(y, x)  # returns in (-pi, pi]
        if  phi < 0:
            phi += 2*np.pi
        return theta, phi

    def test(self):
        S = 0
        for i in range(1000):
            data = self.construct_data()
            y = self.computation_f_last(data[0])
            S += np.dot(y-data[2],y-data[2])
        return S/1000
    
    def algorthim(self,t,gamma0,Epoch,NN,Uni):
        self.Error = []
        self.Time = []
        if Uni == "time":
            DATA_time = []
            for data in DATA[:Epoch]:
                an_theta, an_phi = data[0]
                X = np.array([math.sin(an_theta) * math.cos(an_phi), math.sin(an_theta) * math.sin(an_phi), math.cos(an_theta)])
                DATA_time.append([X, data[1], data[2]])
        start = time.perf_counter()
        self.t = t
        self.gamma0 = gamma0
        for epoch in range(Epoch):
            if Uni == "time":
                Z = DATA_time[epoch]
            else:
                Z = self.construct_data()
            self.updata(Z[0], Z[1], epoch+1)
            if epoch + 1 in NN:
                if Uni == "time":
                    end = time.perf_counter()
                    runTime = end - start
                    self.Time.append(runTime)
                    #print(epoch,'time',runTime)
                elif Uni == 'error':
                    err = self.test()
                    self.Error.append(err)
                    print(epoch + 1,'err',err)
        return self.Error
            
        
        

In [ ]:
d = 3
number = 11
N = [int(10**(i/5))for i in range(1,20)]
s = 1.
r=1.
gamma0 = 1.6
t = 2*r/(2*r+1)
theta = 1/(2*s*(2*r+1))
Sum_epoch = int(N[-1])
sigma = 1.
T = 5
Loss = ['Cauchy', 'Huber', 'Welsch']
Kernel = ['Matern52','Gauss','Matern32']
for loss in Loss:
    for kernel in Kernel:
        err = []
        err_ave = []
        for i in range(T):
            kernel_SGD = Kernel_SGD(1,d,10**(-8*i),loss,kernel,sigma,number,r,s)
            kernel_SGD.algorthim(t,gamma0,Sum_epoch,N,'error')
            err.append(kernel_SGD.Error[:])
        for i in range(len(err[-1])):
            S = 0
            for j in range(T):
                S += err[j][i]
            err_ave.append(S/T)
        if loss == 'least square':
            value = 'least'
        else:
            value = loss
        print('err_'+kernel+'_'+value+'_s_1 =',err_ave)
    print(' ')

In [ ]:
d = 3
number = 10
N = [int(10**(i/5))for i in range(1,20)]
s = 1.
r=1.
gamma0 = 1.6
t = 2*r/(2*r+1)
theta = 1/(2*s*(2*r+1))
Sum_epoch = int(N[-1]+1)
sigma = 1.
T = 10
Loss = ['least square','Cauchy', 'Huber', 'Welsch']
Kernel = ['Matern52','Gauss','Matern32']
for loss in Loss:
    for kernel in Kernel:
        Time = []
        Time_ave = []
        for i in range(T):
            kernel_SGD = Kernel_SGD(1,d,10**(-8*i),loss,kernel,sigma,number,r,s)
            kernel_SGD.algorthim(t,gamma0,Sum_epoch,N,'time')
            Time.append(kernel_SGD.Time[:])
        for i in range(len(Time[-1])):
            S = 0
            for j in range(T):
                S += Time[j][i]
            Time_ave.append(S/T)
        if loss == 'least square':
            value = 'least'
        else:
            value = loss
        print('time_'+kernel+'_'+value+'_s_1 =',Time_ave)
    print(' ')